In [1]:
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable


In [33]:
import os
from pathlib import Path
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [35]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\swlee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [69]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))
# To keep the negation words
for w in ['no''not''none''neither''never''nobody''nothing''nowhere''doesn\'t''isn\'t''wasn\'t''shouldn\'t''won\'t''can\'t''couldn\'t''don\'t''haven\'t''hasn\'t''hadn\'t''aren\'t''weren\'t''wouldn\'t''daren\'t''needn\'t''didn\'t']:
    stop_words.discard(w)

In [71]:
def custom_analyzer(text: str):
    tokens = word_tokenize(text)
    clean = [
        stemmer.stem(word.lower()) 
        for word in tokens 
        if word.isalpha() and word.lower() not in stop_words
    ]

    return clean

In [73]:
def load_reviews(folder_path: Path, label: int):
    texts, labels = [], []
    for file_path in sorted(folder_path.glob("*.txt")):
        with open(file_path, "r", encoding="utf-8") as f:
            texts.append(f.read())
        labels.append(label)
    return texts, labels

In [75]:
def load_dataset(BASE_PATH: str):
    base = Path(BASE_PATH)

    train_pos = base / "train" / "pos"
    train_neg = base / "train" / "neg"
    test_pos  = base / "test" / "pos"
    test_neg  = base / "test" / "neg"

    for p in [train_pos, train_neg, test_pos, test_neg]:
        if not p.exists():
            raise FileNotFoundError(f"Missing folder: {p}")

    X_train_pos, y_train_pos = load_reviews(train_pos, 1)
    X_train_neg, y_train_neg = load_reviews(train_neg, 0)
    X_test_pos,  y_test_pos  = load_reviews(test_pos,  1)
    X_test_neg,  y_test_neg  = load_reviews(test_neg,  0)

    X_train = X_train_pos + X_train_neg
    y_train = y_train_pos + y_train_neg

    X_test  = X_test_pos + X_test_neg
    y_test  = y_test_pos + y_test_neg

    return X_train, y_train, X_test, y_test

In [1]:
# BASE_PATH = "aclImdb"

X_train_texts, y_train, X_test_texts, y_test = load_dataset(BASE_PATH)

NameError: name 'load_dataset' is not defined

In [78]:
print(f"Train size: {len(X_train_texts)}  (pos={sum(y_train)}, neg={len(y_train)-sum(y_train)})")
print(f"Test size : {len(X_test_texts)}   (pos={sum(y_test)}, neg={len(y_test)-sum(y_test)})")

Train size: 25000  (pos=12500, neg=12500)
Test size : 25000   (pos=12500, neg=12500)


In [89]:
def train_and_eval(vectorizer, X_train_texts, y_train, X_test_texts, y_test, title="Model"):
    X_train_vec = vectorizer.fit_transform(X_train_texts)
    X_test_vec  = vectorizer.transform(X_test_texts)

    model = MultinomialNB()
    model.fit(X_train_vec, y_train)

    preds = model.predict(X_test_vec)

    acc = accuracy_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    print(title)
    print(f"\nAccuracy: {acc:.4f}")
    print("Confusion Matrix (rows=true [neg,pos], cols=pred [neg,pos]):")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=["negative", "positive"]))

    return acc

In [83]:
count_vec = CountVectorizer(
    analyzer=custom_analyzer,
    ngram_range=(1, 2),  # good for "not good"
    min_df=2
)

acc_count = train_and_eval(
    count_vec,
    X_train_texts, y_train,
    X_test_texts, y_test,
    title="CountVectorizer"
)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:537: UserWarning: The parameter 'ngram_range' will not be used since 'analyzer' is callable'
  warnings.warn(


CountVectorizer

Accuracy: 0.8180
Confusion Matrix (rows=true [neg,pos], cols=pred [neg,pos]):
[[10945  1555]
 [ 2995  9505]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.79      0.88      0.83     12500
    positive       0.86      0.76      0.81     12500

    accuracy                           0.82     25000
   macro avg       0.82      0.82      0.82     25000
weighted avg       0.82      0.82      0.82     25000



In [84]:
tfidf_vec = TfidfVectorizer(
    analyzer=custom_analyzer,
    ngram_range=(1, 2),
    min_df=2
)

acc_tfidf = train_and_eval(
    tfidf_vec,
    X_train_texts, y_train,
    X_test_texts, y_test,
    title="TfidfVectorizer"
)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:537: UserWarning: The parameter 'ngram_range' will not be used since 'analyzer' is callable'
  warnings.warn(


TfidfVectorizer

Accuracy: 0.8228
Confusion Matrix (rows=true [neg,pos], cols=pred [neg,pos]):
[[10890  1610]
 [ 2819  9681]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.79      0.87      0.83     12500
    positive       0.86      0.77      0.81     12500

    accuracy                           0.82     25000
   macro avg       0.83      0.82      0.82     25000
weighted avg       0.83      0.82      0.82     25000



In [85]:
print("\nSummary:")
print(f"  CountVectorizer accuracy: {acc_count:.4f}")
print(f"  TfidfVectorizer accuracy: {acc_tfidf:.4f}")


Summary:
  CountVectorizer accuracy: 0.8180
  TfidfVectorizer accuracy: 0.8228
